# Uji lintas tahun HWD pada Susenas

Notebook ini menguji kemampuan HWD mengimputasi **tahun survei yang tidak ikut dilatih**:

| Skenario | Data latih | Data uji |
|---|---|---|
| A | 2023 | 2024 |
| B | 2024 | 2025 |
| C | 2023–2024 | 2025 |

Perbedaannya dengan eksperimen Susenas utama:

1. **Parameter praproses hanya dari tahun latih.** Batas outlier 3×IQR, rata-rata, dan simpangan baku dihitung dari tahun latih, lalu diterapkan ke tahun uji.
2. **Window dibentuk per tahun**, sehingga tidak ada window yang berisi campuran tahun latih dan tahun uji.
3. **Mask eksperimen hanya pada tahun uji.** Seluruh nilai teramati pada tahun latih dipakai untuk pelatihan. Posisi uji dibuat di tahun uji dengan `make_mask` yang sama seperti eksperimen utama.

Fungsi `make_mask`, `SusenasDataset`, konfigurasi model, serta loop pelatihan dan inferensi disalin dari notebook Susenas utama agar hasilnya sebanding.

**Cara menjalankan di Kaggle:** tambahkan repo HWD (folder `models/` dan `layers/`) serta berkas data Susenas sebagai *Input*, aktifkan GPU, lalu jalankan satu skenario per sesi melalui `RUN_SCENARIOS`. Checkpoint disimpan setiap 10 epoch. Bila sesi terputus, simpan versi notebook, tambahkan output versi itu sebagai Input di sesi berikutnya, dan pelatihan akan dilanjutkan otomatis.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. KONFIGURASI                                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, sys, glob, time, json, math, shutil, subprocess, warnings, inspect
from types import SimpleNamespace
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', category=RuntimeWarning)

SCENARIOS = {
    'A': {'train': [2023],       'test': 2024},
    'B': {'train': [2024],       'test': 2025},
    'C': {'train': [2023, 2024], 'test': 2025},
}
RUN_SCENARIOS = ['A']            # satu skenario per sesi: ['A'], lalu ['B'], lalu ['C']

MECHANISM      = 'mcar'          # 'mcar' | 'mar' | 'block'
MISSING_RATE   = 0.1
SEED           = 1
SEQ_LEN        = 48
N_SAMPLES      = 10
BATCH          = 32
INFER_BATCH    = 32
REMOVE_OUTLIER = True            # operational mode, sama dengan eksperimen Susenas utama

# Kebijakan epoch
#  'fixed'   : BASE_EPOCHS untuk semua skenario (paling cepat)
#  'matched' : jumlah pembaruan parameter disamakan dengan run penuh (100 epoch × 670 batch)
EPOCH_POLICY = 'fixed'
BASE_EPOCHS  = 100
FULL_UPDATES = 100 * 670

WAVELET, LEVELS, ABLATION, ARCH_V2 = 'db2', 3, None, False
ABL  = ABLATION if ABLATION else 'full'
ARCH = 'v2' if ARCH_V2 else 'v1'
MODE = 'operational' if REMOVE_OUTLIER else 'benchmark'

WORK_DIR = '/kaggle/working'
CKPT_DIR = f'{WORK_DIR}/Checkpoints'
OUT_DIR  = f'{WORK_DIR}/holdout_tahun'
FSUS_OVERRIDE = ''               # isi path berkas Susenas bila pencarian otomatis salah pilih
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print('Skenario yang dijalankan:', RUN_SCENARIOS, '|', MECHANISM, MISSING_RATE, '|', MODE)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. KODE MODEL, PUSTAKA, DAN DATA                                ║
# ╚══════════════════════════════════════════════════════════════════╝
def find_files(patterns, roots=('/kaggle/input', '/kaggle/working')):
    hits = []
    for r in roots:
        for p in patterns:
            hits += glob.glob(os.path.join(r, '**', p), recursive=True)
    return sorted(set(hits))

main_py = find_files(['main_model.py'])
assert main_py, 'models/main_model.py tidak ditemukan. Tambahkan repo HWD sebagai Input.'
CODE_DIR = os.path.dirname(os.path.dirname(main_py[0]))
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
print('Kode HWD:', CODE_DIR)

try:
    import properscoring as ps
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'properscoring'], check=True)
    import properscoring as ps

import torch
from torch.utils.data import Dataset, DataLoader
from torch import optim
from models import main_model, diffusion

src_d = inspect.getsource(diffusion.diff_HWD)
flags = {k: (k in src_d) for k in ['arch_v2', 'branch_merge', '_expand_to_KL',
                                     'no_wavelet', 'no_crosslevel', 'no_conditioning']}
print('Versi diffusion.py:', flags)
if not all(flags.values()):
    print('PERINGATAN: diffusion.py belum versi terbaru. Unggah ulang dari Drive sebelum melanjutkan.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Perangkat:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

skip_words = ('imputed', 'contoh', 'evaluasi', 'ringkasan', 'holdout', 'per_variabel')
cands = [f for f in find_files(['*usenas*.csv', '*usenas*.xlsx'])
         if not any(w in os.path.basename(f).lower() for w in skip_words)]
FSUS = FSUS_OVERRIDE or (cands[0] if cands else None)
assert FSUS, 'Berkas data Susenas tidak ditemukan di /kaggle/input.'
print('Berkas Susenas:', FSUS, '| kandidat lain:', cands[1:4])
df_raw = pd.read_excel(FSUS) if FSUS.lower().endswith(('.xlsx', '.xls')) else pd.read_csv(FSUS, low_memory=False)
print('Data mentah:', df_raw.shape)

FEAT_BASE = ['R101', 'R102', 'R105', 'R301', 'R705', 'R706', 'R707',
             'food', 'nonfood', 'expend', 'kapita']
colmap = {c.lower(): c for c in df_raw.columns}
def col(name):
    return colmap.get(name.lower(), name)
FEAT_COLS = [col(c) for c in FEAT_BASE]
YEAR_COL  = col('tahun')
assert YEAR_COL in df_raw.columns, f'Kolom tahun ({YEAR_COL}) tidak ada di data.'

SKIP_OUTLIER_IDX = [0, 1, 2, 3, 4, 5, 6]     # identitas dan kategorik tidak dicek outliernya
ID_COLS_IDX      = [0, 1]                    # R101, R102 tidak pernah disembunyikan
MAR_COL          = FEAT_COLS.index(col('food'))
BLOCK_LEN        = 4
EXPEND_NAMES     = ['food', 'nonfood', 'expend', 'kapita']
EXPEND_IDX       = [FEAT_COLS.index(col(c)) for c in EXPEND_NAMES]

X_all = df_raw[FEAT_COLS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
YEARS = pd.to_numeric(df_raw[YEAR_COL], errors='coerce').to_numpy()
print('Jumlah baris per tahun:', pd.Series(YEARS).value_counts().sort_index().to_dict())

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. FUNGSI PRAPROSES, WINDOW, MASK, DAN METRIK                   ║
# ║     (tanpa PyTorch)                                              ║
# ╚══════════════════════════════════════════════════════════════════╝
def fit_preprocess(X_train):
    # Batas outlier 3×IQR dan parameter normalisasi dihitung dari tahun latih saja
    fences = {}
    if REMOVE_OUTLIER:
        for j in range(X_train.shape[1]):
            if j in SKIP_OUTLIER_IDX:
                continue
            v = X_train[:, j]
            v = v[~np.isnan(v)]
            if len(v) < 10:
                continue
            q1, q3 = np.percentile(v, 25), np.percentile(v, 75)
            iqr = q3 - q1
            lo, hi = q1 - 3.0 * iqr, q3 + 3.0 * iqr
            if q1 >= 0:
                lo = max(lo, 0.0)
            fences[j] = (lo, hi)
    Xc = apply_fences(X_train, fences)[0]
    means = np.nan_to_num(np.nanmean(Xc, axis=0), nan=0.0)
    stds = np.nan_to_num(np.nanstd(Xc, axis=0), nan=1.0)
    stds[stds == 0] = 1.0
    return fences, means, stds


def apply_fences(X, fences):
    X = X.copy()
    n_removed = 0
    for j, (lo, hi) in fences.items():
        m = ~np.isnan(X[:, j]) & ((X[:, j] < lo) | (X[:, j] > hi))
        n_removed += int(m.sum())
        X[m, j] = np.nan
    return X, n_removed


def normalize(X, means, stds):
    Xn = (X - means) / stds
    Xn[np.isnan(Xn)] = -200.0
    return Xn


def windows_for_years(Xn, years, year_list):
    # Window dibentuk terpisah per tahun agar tidak melintasi batas tahun
    blocks = []
    for y in year_list:
        Z = Xn[years == y]
        n_ok = (len(Z) // SEQ_LEN) * SEQ_LEN
        if n_ok:
            blocks.append(Z[:n_ok].reshape(-1, SEQ_LEN, Z.shape[1]))
    return np.concatenate(blocks, axis=0)


def make_mask(data, gt_mask, mechanism, missing_rate, seed,
              id_cols_idx, mar_col=None, block_len=4):
    # Sama persis dengan make_mask pada notebook Susenas utama
    rng = np.random.default_rng(seed)
    N, L, K = data.shape
    cond_mask = gt_mask.copy()
    obs_idx = np.argwhere(gt_mask == 1)
    obs_idx = obs_idx[~np.isin(obs_idx[:, 2], id_cols_idx)]
    n_hide = int(len(obs_idx) * missing_rate)
    if mechanism == 'mcar':
        pick = obs_idx[rng.choice(len(obs_idx), n_hide, replace=False)]
        for (a, b, c) in pick:
            cond_mask[a, b, c] = 0.0
    elif mechanism == 'mar':
        fval = data[:, :, mar_col]; fobs = gt_mask[:, :, mar_col]
        fsum = (fval * fobs).sum(axis=1); fcnt = fobs.sum(axis=1)
        fmean = np.where(fcnt > 0, fsum / np.maximum(fcnt, 1), np.nan)
        fmean[np.isnan(fmean)] = np.nanmin(fmean)
        rank = (np.argsort(np.argsort(fmean)) + 1).astype(np.float64)
        w_win = rank / rank.sum()
        weight = np.repeat(w_win, L * K).reshape(N, L, K) * gt_mask
        weight[:, :, id_cols_idx] = 0.0
        weight = weight.flatten(); weight = weight / weight.sum()
        u = rng.random(weight.shape)
        keys = np.full(weight.shape, np.inf)           # bobot nol tidak pernah terpilih
        pos = weight > 0
        keys[pos] = -np.log(u[pos] + 1e-12) / weight[pos]
        chosen = np.argpartition(keys, n_hide)[:n_hide]
        flat = cond_mask.flatten(); flat[chosen] = 0.0
        cond_mask = flat.reshape(N, L, K)
    elif mechanism == 'block':
        maskable = [c for c in range(K) if c not in id_cols_idx]
        hidden, guard = 0, 0
        while hidden < n_hide and guard < n_hide * 50:
            guard += 1
            w = rng.integers(0, N)
            c = maskable[rng.integers(0, len(maskable))]
            s = rng.integers(0, max(1, L - block_len + 1))
            for t in range(s, min(s + block_len, L)):
                if gt_mask[w, t, c] == 1 and cond_mask[w, t, c] == 1:
                    cond_mask[w, t, c] = 0.0
                    hidden += 1
    else:
        raise ValueError(f'mechanism tidak dikenal: {mechanism}')
    return cond_mask


def prepare_scenario(train_years, test_year):
    tr_rows = np.isin(YEARS, train_years)
    fences, means, stds = fit_preprocess(X_all[tr_rows])
    X_clean, n_out_total = apply_fences(X_all, fences)
    n_out_test = apply_fences(X_all[YEARS == test_year], fences)[1]
    Xn = normalize(X_clean, means, stds)
    data_tr = windows_for_years(Xn, YEARS, train_years)
    data_te = windows_for_years(Xn, YEARS, [test_year])
    gt_tr = (data_tr != -200).astype(np.float32)
    gt_te = (data_te != -200).astype(np.float32)
    cond_te = make_mask(data_te, gt_te, MECHANISM, MISSING_RATE, SEED,
                        ID_COLS_IDX, mar_col=MAR_COL, block_len=BLOCK_LEN)
    ev_te = (gt_te == 1) & (cond_te == 0)
    info = {'n_window_latih': int(len(data_tr)), 'n_window_uji': int(len(data_te)),
            'n_posisi_uji': int(ev_te.sum()), 'outlier_dihapus_tahun_uji': n_out_test,
            'rate_aktual': float(ev_te.sum() / max(gt_te.sum(), 1))}
    return dict(data_tr=data_tr, gt_tr=gt_tr, data_te=data_te, gt_te=gt_te,
                cond_te=cond_te, ev_te=ev_te, means=means, stds=stds, info=info)


def metrics(true, pred, ev, means, stds):
    d = pred[ev] - true[ev]
    out = {'MAE_z': float(np.abs(d).mean()), 'RMSE_z': float(np.sqrt((d ** 2).mean()))}
    for name, k in zip(EXPEND_NAMES, EXPEND_IDX):
        m = ev[:, :, k]
        if m.sum() == 0:
            continue
        y = true[:, :, k][m] * stds[k] + means[k]
        yh = pred[:, :, k][m] * stds[k] + means[k]
        nz = np.abs(y) > 0
        out[f'MAE_rp_{name}'] = float(np.abs(yh - y).mean())
        out[f'RMSE_rp_{name}'] = float(np.sqrt(((yh - y) ** 2).mean()))
        out[f'MAPE_{name}'] = float(np.mean(np.abs((yh[nz] - y[nz]) / y[nz])) * 100)
    return out

print('Fungsi praproses siap.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. FUNGSI PELATIHAN DAN INFERENSI HWD                           ║
# ║     (sama dengan loop notebook Susenas utama)                    ║
# ╚══════════════════════════════════════════════════════════════════╝
class SusenasDataset(Dataset):
    def __init__(self, data, cond_mask, gt_mask):
        self.data = torch.from_numpy(data).float()
        self.cond_mask = torch.from_numpy(cond_mask).float()
        self.gt_mask = torch.from_numpy(gt_mask).float()
        self.tp = torch.arange(data.shape[1], dtype=torch.float32)
    def __len__(self):
        return self.data.shape[0]
    def __getitem__(self, i):
        return (self.data[i], self.cond_mask[i], self.tp, self.gt_mask[i])


def make_cfg(n_feat, epochs):
    return SimpleNamespace(
        seq_len=SEQ_LEN, enc_in=n_feat, c_out=n_feat,
        d_model=128, e_layers=4, nheads=8, channel=128, proj_t=128,
        residual_layers=4, timeemb=128, featureemb=16,
        diffusion_step_num=50, schedule='quad', beta_start=1e-4, beta_end=0.2,
        epoch_diff=epochs, learning_rate_diff=1e-3,
        mask_ratio_ssl=0.2, avg_mask_len_ssl=3,
        wavelet=WAVELET, levels=LEVELS, batch=BATCH,
        device=DEVICE, missing_rate=MISSING_RATE, seed=SEED, n_samples=N_SAMPLES,
        mechanism=MECHANISM, ablation=ABLATION, arch_v2=ARCH_V2)


def train_hwd(cfg, data_tr, gt_tr, ckpt_path, epochs):
    # Tahun latih dipakai utuh: seluruh nilai teramati menjadi kondisi,
    # target latihan dipilih sendiri oleh masking self-supervised di dalam model.
    np.random.seed(SEED); torch.manual_seed(SEED)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed(SEED)
    loader = DataLoader(SusenasDataset(data_tr, gt_tr.copy(), gt_tr), batch_size=cfg.batch, shuffle=True)
    model = main_model.HWD(cfg).to(cfg.device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate_diff, weight_decay=1e-6)
    p1, p2 = int(0.75 * epochs), int(0.90 * epochs)
    sched = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[p1, p2], gamma=0.1)
    losses, start = [], 0

    src = ckpt_path if os.path.exists(ckpt_path) else None
    if src is None:
        prev = find_files([os.path.basename(ckpt_path)], roots=('/kaggle/input',))
        if prev:
            shutil.copy(prev[0], ckpt_path)
            src = ckpt_path
            print('   Checkpoint sesi sebelumnya disalin dari', prev[0])
    if src:
        ck = torch.load(src, map_location=cfg.device, weights_only=False)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        sched.load_state_dict(ck['scheduler'])
        start = ck['epoch'] + 1
        losses = ck.get('losses', [])
        print(f'   Melanjutkan dari epoch {start}/{epochs}')

    t_train = time.time()
    for epoch in range(start, epochs):
        model.train(); ep = []; t0 = time.time()
        for obs_d, obs_m, obs_tp, gt_m in loader:
            obs_d, obs_m, gt_m = obs_d.to(cfg.device), obs_m.to(cfg.device), gt_m.to(cfg.device)
            optimizer.zero_grad()
            loss = model(obs_d, obs_m, obs_tp.to(cfg.device), gt_m)
            loss.backward(); optimizer.step(); ep.append(loss.item())
        sched.step(); losses.append(float(np.mean(ep)))
        if epoch % 10 == 0 or epoch == epochs - 1:
            torch.save({'epoch': epoch, 'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(), 'scheduler': sched.state_dict(),
                        'losses': losses, 'mechanism': MECHANISM, 'missing_rate': MISSING_RATE,
                        'ablation': ABLATION, 'arch_v2': ARCH_V2, 'wavelet': WAVELET,
                        'levels': LEVELS, 'mode': MODE}, ckpt_path)
            dt = time.time() - t0
            print(f'   Epoch {epoch+1:>4}/{epochs} | {dt:5.1f} dtk | loss {np.mean(ep):.6f} '
                  f'| perkiraan sisa {dt * (epochs - epoch - 1) / 60:.0f} menit')
    return model, losses, time.time() - t_train


def infer_hwd(model, data_te, cond_te, gt_te):
    loader = DataLoader(SusenasDataset(data_te, cond_te, gt_te), batch_size=INFER_BATCH, shuffle=False)
    model.eval()
    crps_sum, npts, med_chunks = 0.0, 0, []
    t0 = time.time()
    with torch.no_grad():
        for b, (obs_d, obs_m, obs_tp, gt_m) in enumerate(loader):
            samples, obs, ev_mask, _, _ = model.evaluate(obs_d, obs_m, obs_tp, gt_m, n_samples=N_SAMPLES)
            med = samples.median(dim=1).values
            m_np = ev_mask.cpu().numpy().astype(bool)
            if m_np.any():
                crps_sum += ps.crps_ensemble(obs.cpu().numpy()[m_np],
                                             np.moveaxis(samples.cpu().numpy(), 1, -1)[m_np]).sum()
                npts += int(m_np.sum())
            med_chunks.append(med.permute(0, 2, 1).cpu().numpy())
            if b % 20 == 0:
                print(f'   Inferensi batch {b+1}/{len(loader)} | {time.time()-t0:.0f} dtk')
    return np.concatenate(med_chunks, axis=0), crps_sum / max(npts, 1), time.time() - t0

print('Fungsi pelatihan dan inferensi siap.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. JALANKAN SKENARIO                                            ║
# ╚══════════════════════════════════════════════════════════════════╝
for sc in RUN_SCENARIOS:
    spec = SCENARIOS[sc]
    tr, te = spec['train'], spec['test']
    tag = f"{sc}_{'-'.join(map(str, tr))}to{te}_{MECHANISM}_mr{MISSING_RATE}"
    print(f"\n{'=' * 66}\n  Skenario {sc}: latih {tr} → uji {te}  |  {MECHANISM} {MISSING_RATE}  |  {MODE}\n{'=' * 66}")

    S = prepare_scenario(tr, te)
    print('  ', S['info'])
    n_batch = math.ceil(S['info']['n_window_latih'] / BATCH)
    epochs = BASE_EPOCHS if EPOCH_POLICY == 'fixed' else math.ceil(FULL_UPDATES / n_batch)
    print(f'   {n_batch} batch per epoch × {epochs} epoch = {n_batch * epochs:,} pembaruan '
          f'(run penuh: {FULL_UPDATES:,})')

    cfg = make_cfg(S['data_tr'].shape[2], epochs)
    ckpt = f'{CKPT_DIR}/hwd_susenas_holdout_{tag}_{ABL}_{ARCH}_{WAVELET}_{MODE}_e{epochs}.pt'
    model, losses, t_train = train_hwd(cfg, S['data_tr'], S['gt_tr'], ckpt, epochs)
    imputed, crps, t_inf = infer_hwd(model, S['data_te'], S['cond_te'], S['gt_te'])

    res = {'skenario': sc, 'latih': '-'.join(map(str, tr)), 'uji': te, 'epoch': epochs,
           'mekanisme': MECHANISM, 'missing_rate': MISSING_RATE, 'mode': MODE}
    res.update(S['info'])
    res.update({'metode': 'HWD', 'CRPS_z': float(crps),
                'detik_latih': round(t_train, 1), 'detik_inferensi': round(t_inf, 1)})
    res.update(metrics(S['data_te'], imputed, S['ev_te'], S['means'], S['stds']))
    rows = [res]

    # Pembanding sederhana pada posisi uji yang sama (statistik dari tahun latih)
    flat_tr = np.where(S['gt_tr'] == 1, S['data_tr'], np.nan).reshape(-1, S['data_tr'].shape[2])
    for name, stat in [('Mean', np.nanmean(flat_tr, axis=0)), ('Median', np.nanmedian(flat_tr, axis=0))]:
        pred = np.broadcast_to(np.nan_to_num(stat), S['data_te'].shape)
        r = {k: res[k] for k in ('skenario', 'latih', 'uji', 'mekanisme', 'missing_rate', 'mode')}
        r['metode'] = name
        r.update(metrics(S['data_te'], pred, S['ev_te'], S['means'], S['stds']))
        rows.append(r)

    pd.DataFrame(rows).to_csv(f'{OUT_DIR}/hasil_{tag}.csv', index=False)
    np.save(f'{OUT_DIR}/data_{tag}_norm.npy', S['data_te'])
    np.save(f'{OUT_DIR}/condmask_{tag}.npy', S['cond_te'])
    np.save(f'{OUT_DIR}/evalmask_{tag}.npy', S['ev_te'])
    np.save(f'{OUT_DIR}/imputed_{tag}_norm.npy', imputed)
    np.save(f'{OUT_DIR}/means_{tag}.npy', S['means'])
    np.save(f'{OUT_DIR}/stds_{tag}.npy', S['stds'])
    json.dump(losses, open(f'{OUT_DIR}/losses_{tag}.json', 'w'))

    print(f"\n   HWD    MAE_z {res['MAE_z']:.4f} | RMSE_z {res['RMSE_z']:.4f} | CRPS_z {res['CRPS_z']:.4f}")
    for r in rows[1:]:
        print(f"   {r['metode']:<6} MAE_z {r['MAE_z']:.4f} | RMSE_z {r['RMSE_z']:.4f}")
    print(f'   Waktu: latih {t_train/3600:.2f} jam, inferensi {t_inf/3600:.2f} jam')
    print(f'   Hasil tersimpan di {OUT_DIR}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. RINGKASAN SEMUA SKENARIO YANG SUDAH SELESAI                  ║
# ║     (membaca hasil sesi ini dan sesi sebelumnya di /kaggle/input) ║
# ╚══════════════════════════════════════════════════════════════════╝
files = find_files(['hasil_*to*_*.csv'], roots=(OUT_DIR, '/kaggle/input'))
if not files:
    print('Belum ada hasil.')
else:
    allres = pd.concat([pd.read_csv(f) for f in files]).drop_duplicates(subset=['skenario', 'metode'], keep='last')
    allres = allres.sort_values(['skenario', 'metode'])
    cols = ['skenario', 'latih', 'uji', 'metode', 'MAE_z', 'RMSE_z', 'CRPS_z'] + \
           [f'MAPE_{n}' for n in EXPEND_NAMES]
    print(allres[[c for c in cols if c in allres.columns]].round(4).to_string(index=False))
    rp = ['skenario', 'metode'] + [f'{m}_rp_{n}' for n in EXPEND_NAMES for m in ('MAE', 'RMSE')]
    print('\nGalat pengeluaran (rupiah):')
    print(allres[[c for c in rp if c in allres.columns]].round(0).to_string(index=False))
    allres.to_csv(f'{OUT_DIR}/ringkasan_holdout.csv', index=False)
    print(f'\nTersimpan: {OUT_DIR}/ringkasan_holdout.csv')

## Catatan untuk penulisan

- **Skala Z tidak sebanding antarskenario.** Setiap skenario memakai rata-rata dan simpangan baku dari tahun latihnya sendiri, sehingga satuan Z-score-nya berbeda. Bandingkan antarskenario, maupun dengan eksperimen utama, memakai MAE rupiah dan MAPE per variabel pengeluaran.
- **Pembanding eksperimen utama.** MAE dan MAPE per tahun dari eksperimen utama sudah tersedia di scatter 2024 dan 2025. Kalau angka lintas tahun berdekatan dengan angka itu, HWD terbukti mampu mengimputasi tahun yang tidak ikut dilatih.
- **Kebijakan epoch.** Dengan `EPOCH_POLICY = 'fixed'`, skenario A dan B hanya mendapat sepertiga jumlah pembaruan parameter dibanding run penuh, dan skenario C dua pertiganya. Tulis kebijakan yang dipakai di naskah.
- **Pembanding Mean dan Median** dihitung dari tahun latih dan dinilai pada posisi uji yang sama. Hot-deck bisa ditambahkan dengan notebook pembanding memakai berkas `data_`, `condmask_`, dan `evalmask_` yang tersimpan.